# Import Libraries and Data Importing


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/workspaces/Reserving-using-Agentic-AI/data/comauto_pos.csv")

# Data Exploration

In [3]:
df.head()

,GRCODE,GRNAME,AccidentYear,DevelopmentYear,DevelopmentLag,IncurLoss_C,CumPaidLoss_C,BulkLoss_C,EarnedPremDIR_C,EarnedPremCeded_C,EarnedPremNet_C,Single,PostedReserve97_C
0,266,Public Underwriters Grp,1988,1988,1,0,0,0,0,0,0,0,932
1,266,Public Underwriters Grp,1988,1989,2,0,0,0,0,0,0,0,932
2,266,Public Underwriters Grp,1988,1990,3,0,0,0,0,0,0,0,932
3,266,Public Underwriters Grp,1988,1991,4,0,0,0,0,0,0,0,932
4,266,Public Underwriters Grp,1988,1992,5,0,0,0,0,0,0,0,932


In [4]:
df['GRCODE'].unique()

array([  266,   337,   353,   388,   460,   620,   655,   671,   715,
         833,   965,  1066,  1090,  1279,  1538,  1716,  1767,  2003,
        2135,  2143,  2208,  2569,  2623,  2712,  3131,  3240,  3492,
        4839,  5185,  5320,  5690,  5940,  6408,  6459,  6777,  6807,
        6947,  7080,  8079,  8281,  8427,  8559,  8672,  9466, 10019,
       10022, 10048, 10074, 10100, 10308, 10561, 10790, 10859, 10894,
       11037, 11118, 11126, 11150, 11231, 11460, 12866, 13420, 13439,
       13501, 13528, 13587, 13641, 13889, 13943, 14044, 14176, 14257,
       14311, 14320, 14370, 14508, 14974, 15024, 15199, 15407, 15792,
       15911, 15997, 16411, 16748, 17299, 17884, 18163, 18309, 18380,
       18538, 18686, 18767, 18791, 19020, 19780, 20451, 20690, 21172,
       21270, 22390, 23574, 23663, 25275, 25950, 26077, 26433, 26797,
       26905, 27022, 27065, 27499, 27955, 27980, 28258, 28436, 28535,
       28550, 28886, 29297, 29378, 29440, 31550, 31810, 32301, 32514,
       32670, 32743,

In [5]:
df_company = df[df['GRCODE'] == 13587]

# Functions

In [ ]:
def build_triangle( df, index_col, column_col, value_col, aggfunc="sum"):
    """
    Builds a triangle from a long-form DataFrame.
    """
    return df.pivot_table(
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc=aggfunc
    )

def incremental_to_cumulative(triangle: pd.DataFrame) -> pd.DataFrame:
    """
    Convert incremental loss triangle to cumulative loss triangle.
    Also ensures columns are typed as integers.
    """
    cumulative = triangle.cumsum(axis=1)
    cumulative.columns = cumulative.columns.astype(int)
    return cumulative

def calculate_age_to_age_factors(cumulative_triangle):
    """
    Calculate individual age-to-age factors for every accident year.
    Parameters
    ----------
    cumulative_triangle : pd.DataFrame
        Cumulative loss triangle
    Returns
    -------
    pd.DataFrame
        Age-to-age factor matrix
    """

    factors = pd.DataFrame(index=cumulative_triangle.index)
    for col in range(cumulative_triangle.shape[1] - 1):
        current = cumulative_triangle.iloc[:, col]
        nxt = cumulative_triangle.iloc[:, col + 1]
        factors[f"{col+1}->{col+2}"] = nxt / current
    return factors

def plot_age_to_age(age_to_age_triangle):

    for col in age_to_age_triangle.columns:

        plt.figure(figsize=(8,4))

        plt.plot(
            age_to_age_triangle.index,
            age_to_age_triangle[col],
            marker="o"
        )

        plt.title(f"Age-to-Age Factor {col}")
        plt.xlabel("Accident Year")
        plt.ylabel("Factor")
        plt.grid(True)

        plt.show()

def calculate_average_factors(age_to_age_triangle):

    results = {}

    for col in age_to_age_triangle.columns:

        factors = age_to_age_triangle[col].dropna()

        simple = factors.mean()

        geometric = np.prod(factors) ** (1 / len(factors))

        if len(factors) >= 3:
            medial = (
                factors.sort_values()
                .iloc[1:-1]
                .mean()
            )
        else:
            medial = simple

        results[col] = {
            "Simple": simple,
            "Geometric": geometric,
            "Medial": medial
        }

    return pd.DataFrame(results)

def plot_average_factors(avg_factors):

    plt.figure(figsize=(10,5))

    for avg_type in avg_factors.index:

        plt.plot(
            avg_factors.columns,
            avg_factors.loc[avg_type],
            marker='o',
            label=avg_type
        )

    plt.xlabel("Development Period")
    plt.ylabel("Factor")
    plt.title("Average Development Factors")
    plt.legend()
    plt.grid(True)

    plt.show()

def select_ldfs(avg_factors):

    print("Select LDF Method:")
    print("1. Simple Average")
    print("2. Geometric Average")
    print("3. Medial Average")

    choice = input("Enter choice (1-3): ")

    method_map = {
        "1": "Simple",
        "2": "Geometric",
        "3": "Medial"
    }

    selected_ldfs = avg_factors.loc[method_map[choice]].copy()

    tail_input = input(
        "Enter Tail Factor (press Enter for 1.0): "
    )

    tail_factor = (
        float(tail_input)
        if tail_input.strip()
        else 1.0
    )

    selected_ldfs["Tail"] = tail_factor

    return selected_ldfs

def calculate_cdfs(selected_ldfs):

    cdfs = []

    factors = selected_ldfs.values

    for i in range(len(factors)):
        cdfs.append(np.prod(factors[i:]))

    return pd.Series(cdfs)

def project_ultimates(triangle, cdfs):

    cdfs = cdfs.copy()
    cdfs.index = triangle.columns

    results = []

    for ay in triangle.index:

        row = triangle.loc[ay].dropna()

        latest_age = row.index[-1]
        latest_value = row.iloc[-1]

        cdf = cdfs.loc[latest_age]

        ultimate = latest_value * cdf

        results.append({
            "AccidentYear": ay,
            "LatestAge": latest_age,
            "LatestValue": latest_value,
            "CDF": cdf,
            "Ultimate": ultimate
        })

    return pd.DataFrame(results)

def calculate_unpaid_claim_estimate(ultimate_df):

    reserve_df = ultimate_df.copy()

    reserve_df["TotalReserveEstimate"] = (
        reserve_df["Ultimate"]
        - reserve_df["LatestValue"]
    )

    return reserve_df

def calculate_total_reserve(reserve_df):

    total_reserve = reserve_df["Reserve"].sum()
    return total_reserve

def calculate_case_outstanding(paid_ultimate_df,incurred_ultimate_df):
    
    result = pd.DataFrame()
    
    result['AccidentYear'] = paid_ultimate_df["AccidentYear"]
    result["PaidLatest"] = paid_ultimate_df["LatestValue"]
    result["IncurredLatest"] = incurred_ultimate_df["LatestValue"]
    result["CaseOutstanding"] = (result["IncurredLatest"]-result["PaidLatest"])

    return result

def calculate_ibnr(selected_ultimate_df,incurred_ultimate_df):

    result = pd.DataFrame()

    result["AccidentYear"] = selected_ultimate_df["AccidentYear"]
    result["SelectedUltimate"] = selected_ultimate_df["Ultimate"]
    result["CurrentIncurred"] = incurred_ultimate_df["LatestValue"]
    result["IBNR"] = result["SelectedUltimate"] - result["CurrentIncurred"]
    return result

def reserve_summary( paid_ultimate_df, incurred_ultimate_df, selected_ultimate_df):

    result = pd.DataFrame()

    result["AccidentYear"] = selected_ultimate_df["AccidentYear"]
    result["Paid"] = paid_ultimate_df["LatestValue"]
    result["Incurred"] = incurred_ultimate_df["LatestValue"]
    result["SelectedUltimate"] = selected_ultimate_df["Ultimate"]
    result["CaseReserve"] = result["Incurred"] - result["Paid"]
    result["IBNR"] = result["SelectedUltimate"] - result["Incurred"]
    result["TotalReserve"] = result["SelectedUltimate"] - result["Paid"]
    result["Check"] = result["CaseReserve"] + result["IBNR"]
    result["Difference"] = result["TotalReserve"] - result["Check"]
    result['Status'] = np.where(
            np.isclose(result["Difference"], 0, atol=1),
            "Reconciles",
            "Different Ultimate Selected"
    )
    result["PaidCLUltimate"] = paid_ultimate_df["Ultimate"]
    result["IncurredCLUltimate"] = incurred_ultimate_df["Ultimate"]

    return result

# Data Preparation


In [7]:
paid = build_triangle(df_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='CumPaidLoss_C',aggfunc="sum" )
incurred = build_triangle(df_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='IncurLoss_C',aggfunc="sum" )

This is a fully squared triangle from CAS. So, this is being converted back to a triangular format

In [8]:
df_masked=df[df['DevelopmentYear']<=1997]
df_masked_company = df_masked[df_masked['GRCODE'] == 13587]
paid_masked = build_triangle(df_masked_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='CumPaidLoss_C',aggfunc="sum" )
incurred_masked = build_triangle(df_masked_company,index_col ='AccidentYear',column_col = 'DevelopmentLag',value_col='IncurLoss_C',aggfunc="sum" )

In [9]:
#plot_age_to_age(paid_age_to_age_factors)
#plot_age_to_age(incurred_age_to_age_factors)

# One final function for full Chain Ladder Method

In [10]:
def run_chain_ladder(triangle,method="Simple",tail_factor=1.0,show_plots=True):
    age_to_age = calculate_age_to_age_factors(triangle)

    if show_plots:
        plot_age_to_age(age_to_age)

    avg_ldfs = calculate_average_factors(age_to_age)
    print("\nAverage LDFs")
    display(avg_ldfs)

    if show_plots:
        plot_average_factors(avg_ldfs)

    # Select LDFs
    selected_ldfs = avg_ldfs.loc[method].copy()
    selected_ldfs["Tail"] = tail_factor
    print("\nSelected LDFs")
    display(selected_ldfs)

    cdfs = calculate_cdfs(selected_ldfs)
    print("\nCDFs")
    display(cdfs)

    # Ultimate Projection
    ultimate_df = project_ultimates( triangle, cdfs)
    print("\nUltimate Projection")
    display(ultimate_df)
    
    # Reserves
    reserve_df = calculate_unpaid_claim_estimate(ultimate_df)
    print("\nReserve Projection")
    display(reserve_df)
    # Total Reserve
    total_reserve = calculate_total_reserve(reserve_df)
    print(f"\nTotal Reserve = {total_reserve:,.2f}")
    return {
        "age_to_age": age_to_age,
        "avg_ldfs": avg_ldfs,
        "selected_ldfs": selected_ldfs,
        "cdfs": cdfs,
        "ultimate_df": ultimate_df,
        "reserve_df": reserve_df,
        "total_reserve": total_reserve
    }

In [ ]:
paid_results = run_chain_ladder(
    paid_masked,
    method="Simple",
    tail_factor=1.0,
    show_plots=False
)

In [ ]:
incurred_results = run_chain_ladder(
    incurred_masked,
    method="Simple",
    tail_factor=1.0,
    show_plots=False
)

In [ ]:
paid_ultimate_df = paid_results["ultimate_df"]

incurred_ultimate_df = incurred_results["ultimate_df"]

selected_ultimate_df = paid_ultimate_df.copy() 

summary = reserve_summary(
    paid_ultimate_df,
    incurred_ultimate_df,
    selected_ultimate_df      # selected ultimate
)

display(summary)

So which ultimate is "correct"?

There is no single observed ultimate.

You may have:

| Method      | Ultimate |
| ----------- | -------: |
| Paid CL     |      80M |
| Incurred CL |      78M |
| BF          |      75M |
| Cape Cod    |      77M |


The actuary reviews all indications and selects a reasonable ultimate.

Friedland explicitly notes:

The actuary will ordinarily examine the indications of more than one method when estimating reserves.

This is exactly why.

They first calculate:
Ultimate Loss

and then:
Reserve=Ultimate−Known Amount

where the "known amount" depends on the method:

Paid CL → Ultimate − Paid
Incurred CL → Ultimate − Paid (total reserve)
Incurred CL → Ultimate − Incurred (IBNR)

Current Incurred - Paid = Case Reserve
IBNR = Selected Ultimate − Current Incurred